# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/krihna7/flyrank-ml-internship.git /content/flyrank-ML-internship

fatal: destination path '/content/flyrank-ML-internship' already exists and is not an empty directory.


In [ ]:

import os

repo = "/content/flyrank-ML-internship"

print("Repo exists:", os.path.exists(repo))
print("\nTop-level files/folders:")

for item in os.listdir(repo):
    print(item)

Repo exists: True

Top-level files/folders:
data
GUIDE.md
notebooks
submission
skills
DATA_USE.md
.github
.git
scripts
SETUP.md
requirements.txt
CLAUDE.md
outputs
LICENSE
README.md
docs
AGENTS.md
.gitignore
work


In [ ]:
import os

print("Notebooks:")
for root, dirs, files in os.walk("/content/flyrank-ML-internship/work/notebooks"):
    for f in files:
        if f.endswith(".ipynb"):
            print(os.path.join(root, f))

print("\nOutputs:")
for root, dirs, files in os.walk("/content/flyrank-ML-internship/work/outputs"):
    for f in files:
        print(os.path.join(root, f))

Notebooks:
/content/flyrank-ML-internship/work/notebooks/w01_research_question.ipynb
/content/flyrank-ML-internship/work/notebooks/w07_action_playbook.ipynb
/content/flyrank-ML-internship/work/notebooks/w03_feature_leakage_check.ipynb
/content/flyrank-ML-internship/work/notebooks/w04_signal_audit.ipynb
/content/flyrank-ML-internship/work/notebooks/w03_data_contract.ipynb
/content/flyrank-ML-internship/work/notebooks/capstone.ipynb
/content/flyrank-ML-internship/work/notebooks/w02_ml_task_framing.ipynb
/content/flyrank-ML-internship/work/notebooks/w04_baseline_score.ipynb
/content/flyrank-ML-internship/work/notebooks/w06_validation_audit.ipynb
/content/flyrank-ML-internship/work/notebooks/w05_model.ipynb

Outputs:


In [ ]:
import json
import os

path = "/content/flyrank-ML-internship/work/notebooks/w05_model.ipynb"

with open(path, "r", encoding="utf-8") as f:
    nb = json.load(f)

print("Total cells:", len(nb["cells"]))

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))

    print("\n" + "=" * 80)
    print(f"CELL {i} | {cell['cell_type']}")
    print("=" * 80)
    print(source[:4000])

Total cells: 25

CELL 0 | markdown
# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krihna7/flyrank-ML-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

CELL 1 | code
!git clone https://github.com/krihna7/flyrank-ml-internship.git /content/flyrank-ML-internship

CELL 2 | code
from pathlib import Path

repo = Path("/content/flyrank-ML-internship")

print("Repo exists:", repo.exists())

print("\nNotebooks:")
for p in (repo / "work" / "notebooks").glob("*.ipynb"):
    print(p)

print("\nOutputs:")
output_dir = repo / "work" / "outputs"
if output_dir.exists():
    for p in output_dir.iterdir():
        print(p.name

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two paper findings + my methodology questions

### Finding 1 — The Freshness Multiplier

The FlyRank research paper reports that pages updated 31–90 days ago had a 5.43:1 growth-to-decline ratio. It also reports higher observed Health Scores and impressions for recently refreshed pages compared with older stale pages.

**Methodology question:** Where does the outcome label come from, and does the comparison establish that refreshing the page caused the improvement? I would want to know whether the compared groups were similar in factors such as prior visibility, page age, and other characteristics. I would treat this as an observed relationship unless the validation design supports a causal conclusion.

### Finding 2 — Reader Engagement and Search Visibility

The FlyRank research paper reports that pages with higher scroll depth and reader engagement have higher observed Health Scores, including a reported 16.1-point difference between the highlighted engagement groups.

**Methodology question:** Is the outcome independent of the variables used in the comparison, and does the validation design control for other factors that could explain the relationship? I would want to check whether engagement measures overlap with the Health Score definition and whether factors such as page age and existing search visibility are accounted for. I would treat this as an observed association, not proof that increasing engagement causes better search performance.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

I will validate the Week-5 Random Forest using a grouped train/test split by client. This is more honest than a random row-level split because content from the same client could otherwise appear in both training and test data.

The grouped split keeps test clients unseen during training. I will compare the Week-4 baseline and Random Forest using the same held-out test set and the same Precision@50 metric.

The goal is to measure whether the model provides directional improvement for the defined proxy target, not to claim that it will generalize to every client or that the recommended action will cause future SEO improvement.

In [ ]:
# ML-09 Section 2 — Honest grouped validation

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

REPO = Path("/content/flyrank-ML-internship")
DATA_PATH = REPO / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# Target
df["target_refresh"] = (
    pd.to_numeric(df["trend_pct"], errors="coerce") < -10
).astype(int)

# Final Week-5 feature set
features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "pageviews_90d",
    "users_90d",
    "engaged_sessions_90d",
    "engagement_rate",
    "days_since_last_update"
]

features = [c for c in features if c in df.columns]

print("\nFeatures used:")
print(features)

# Client grouping
client_candidates = ["client_hash_id", "client_id"]

client_column = next(
    (c for c in client_candidates if c in df.columns),
    None
)

assert client_column is not None, "Client grouping column not found."

print("Grouping column:", client_column)

# Convert features to numeric
for col in features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=[client_column]).copy()

# Grouped 80/20 split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=df[client_column])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df[client_column].unique())
test_clients = set(test_df[client_column].unique())

overlap = train_clients.intersection(test_clients)

print("\nHONEST GROUPED SPLIT")
print("=" * 50)
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(overlap))

assert len(overlap) == 0

# Training-only median imputation
for col in features:
    median_value = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

# Train Random Forest
X_train = train_df[features]
y_train = train_df["target_refresh"]

X_test = test_df[features]
y_test = test_df["target_refresh"]

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

test_df["model_probability"] = rf.predict_proba(X_test)[:, 1]

# Precision@50
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores)
    top_k = order[:k]

    return y_true[top_k].mean()

honest_p50 = precision_at_k(
    y_test.values,
    test_df["model_probability"].values,
    k=50
)

print("\nHONEST RANDOM FOREST RESULT")
print("=" * 50)
print(f"Precision@50: {honest_p50:.2%}")

# ML-09 — Week-4 baseline vs honest Random Forest
# Both evaluated on the same held-out test set

baseline_test = test_df.copy()

# Week-4 baseline components
baseline_test["visibility_score"] = (
    baseline_test["impressions_90d"]
    .rank(method="average", pct=True)
)

baseline_test["staleness_score"] = (
    baseline_test["days_since_last_update"]
    .rank(method="average", pct=True)
)

# Week-4 baseline score
baseline_test["baseline_score"] = (
    0.70 * baseline_test["visibility_score"]
    + 0.30 * baseline_test["staleness_score"]
)

# Baseline Precision@50
baseline_p50 = precision_at_k(
    baseline_test["target_refresh"].values,
    baseline_test["baseline_score"].values,
    k=50
)

# Comparison
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest — honest grouped split"
    ],
    "Precision@50": [
        baseline_p50,
        honest_p50
    ]
})

comparison["Precision@50_percent"] = (
    comparison["Precision@50"] * 100
).round(2)

improvement_pp = (
    honest_p50 - baseline_p50
) * 100

print("BEFORE / AFTER VALIDATION")
print("=" * 50)
display(comparison)

print(
    f"Measured difference: {improvement_pp:+.2f} percentage points"
)

Dataset shape: (30000, 44)

Features used:
['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'sessions_90d', 'pageviews_90d', 'users_90d', 'engaged_sessions_90d', 'engagement_rate', 'days_since_last_update']
Grouping column: client_id

HONEST GROUPED SPLIT
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0

HONEST RANDOM FOREST RESULT
Precision@50: 80.00%
BEFORE / AFTER VALIDATION


,method,Precision@50,Precision@50_percent
0,Week-4 baseline,0.52,52.0
1,Random Forest — honest grouped split,0.80,80.0


Measured difference: +28.00 percentage points


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

I audited the final Week-5 feature set for target and validation leakage.

The target is a proxy defined as `trend_pct < -10`, and `trend_pct` is not included in the model features. The model therefore does not directly use the variable that defines the target.

The final feature set contains observed performance, engagement, ranking, and freshness signals. No client identifier, content identifier, or target column is used as a predictive feature.

I also identified a preprocessing issue in the original Week-5 workflow: missing-value medians were calculated before the train/test split. For this validation audit, the grouped split is performed first and missing values are filled using medians calculated from the training data only.

The grouped validation prevents the same client from appearing in both training and test data. The observed 80.0% Precision@50 therefore comes from a test set containing clients that were not present during training.

The remaining limitation is that the target is a rule-derived proxy based on `trend_pct`, rather than a measured post-refresh business outcome. The model should therefore be treated as decision-support and the result as directional evidence of predictive usefulness, not proof that a refresh will improve future SEO performance.

In [ ]:
# ML-09 Section 3 — Leakage audit

print("LEAKAGE AUDIT")
print("=" * 50)

# Target and features
target_column = "target_refresh"

print("Target:", target_column)
print("Target source: trend_pct < -10")

print("\nFinal model features:")
for feature in features:
    print(" -", feature)

# Direct target leakage check
direct_target_leakage = target_column in features
trend_leakage = "trend_pct" in features

print("\nDirect target column used as feature:", direct_target_leakage)
print("trend_pct used directly as feature:", trend_leakage)

# Identifier leakage check
identifier_features = [
    col for col in features
    if col in ["client_id", "client_hash_id", "content_id", "content_hash_id"]
]

print("\nIdentifier features found:")
print(identifier_features)

# Client overlap check
train_clients = set(train_df[client_column].unique())
test_clients = set(test_df[client_column].unique())

client_overlap = train_clients.intersection(test_clients)

print("\nTrain clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

# Summary checks
print("\nAUDIT RESULT")
print("=" * 50)

if not direct_target_leakage:
    print("✓ Target column is not used as a feature.")

if not trend_leakage:
    print("✓ trend_pct is not directly used as a feature.")

if len(identifier_features) == 0:
    print("✓ No client/content identifier is used as a feature.")

if len(client_overlap) == 0:
    print("✓ No client appears in both train and test.")

print("✓ Test-set imputation uses training-set medians only.")
print("✓ Target is a rule-derived proxy, not a post-refresh outcome.")

LEAKAGE AUDIT
Target: target_refresh
Target source: trend_pct < -10

Final model features:
 - impressions_90d
 - clicks_90d
 - ctr
 - avg_position
 - sessions_90d
 - pageviews_90d
 - users_90d
 - engaged_sessions_90d
 - engagement_rate
 - days_since_last_update

Direct target column used as feature: False
trend_pct used directly as feature: False

Identifier features found:
[]

Train clients: 25
Test clients: 7
Client overlap: 0

AUDIT RESULT
✓ Target column is not used as a feature.
✓ trend_pct is not directly used as a feature.
✓ No client/content identifier is used as a feature.
✓ No client appears in both train and test.
✓ Test-set imputation uses training-set medians only.
✓ Target is a rule-derived proxy, not a post-refresh outcome.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Original claim

The Random Forest improved Precision@50 from 52.0% for the Week-4 baseline to 80.0%, a measured improvement of 28.0 percentage points.

### Rewritten safe claim

On the evaluated 30,000-row dataset, the Random Forest recorded 80.0% Precision@50 under a client-grouped 80/20 validation split, compared with 52.0% for the Week-4 baseline on the same held-out test set. This is a measured improvement of 28.0 percentage points and provides directional evidence that the model can support content-prioritization decisions for the defined proxy target.

This result should be treated as decision-support evidence rather than proof of general performance across all clients. The target is a rule-derived proxy based on `trend_pct < -10`, not a measured post-refresh outcome, so the result does not establish that refreshing a recommended page will improve future SEO performance.

In [ ]:
# ML-09 Section 4 — Final measured claim

print("FINAL VALIDATION CLAIM")
print("=" * 50)

print(f"Week-4 baseline Precision@50: {baseline_p50:.2%}")
print(f"Random Forest Precision@50: {honest_p50:.2%}")

improvement_pp = (honest_p50 - baseline_p50) * 100

print(
    f"Measured improvement: {improvement_pp:+.2f} percentage points"
)

print("\nValidation design:")
print("- Grouped by client")
print("- 80/20 train/test split")
print("- Test clients unseen during training")

print("\nInterpretation:")
print("- Observed and measured result")
print("- Directional evidence")
print("- Decision-support use case")
print("- Proxy target, not causal outcome")


# ML-09 — Real failure examples

test_df["predicted_refresh"] = (
    test_df["model_probability"] >= 0.50
).astype(int)

test_df["error_type"] = np.select(
    [
        (test_df["predicted_refresh"] == 1) &
        (test_df["target_refresh"] == 1),

        (test_df["predicted_refresh"] == 1) &
        (test_df["target_refresh"] == 0),

        (test_df["predicted_refresh"] == 0) &
        (test_df["target_refresh"] == 1),

        (test_df["predicted_refresh"] == 0) &
        (test_df["target_refresh"] == 0)
    ],
    [
        "true_positive",
        "false_positive",
        "false_negative",
        "true_negative"
    ],
    default="unknown"
)

print("ERROR COUNTS")
display(
    test_df["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

print("\nTOP FALSE POSITIVES")

display(
    test_df[
        test_df["error_type"] == "false_positive"
    ]
    .sort_values("model_probability", ascending=False)
    [
        ["content_id", "model_probability",
         "target_refresh"] + features
    ]
    .head(5)
)

print("\nTOP FALSE NEGATIVES")

display(
    test_df[
        test_df["error_type"] == "false_negative"
    ]
    .sort_values("model_probability", ascending=False)
    [
        ["content_id", "model_probability",
         "target_refresh"] + features
    ]
    .head(5)
)

FINAL VALIDATION CLAIM
Week-4 baseline Precision@50: 52.00%
Random Forest Precision@50: 80.00%
Measured improvement: +28.00 percentage points

Validation design:
- Grouped by client
- 80/20 train/test split
- Test clients unseen during training

Interpretation:
- Observed and measured result
- Directional evidence
- Decision-support use case
- Proxy target, not causal outcome
ERROR COUNTS


,error_type,count
0,true_positive,2576
1,false_positive,1335
2,true_negative,1240
3,false_negative,1012



TOP FALSE POSITIVES


,content_id,model_probability,target_refresh,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,pageviews_90d,users_90d,engaged_sessions_90d,engagement_rate,days_since_last_update
21559,content_bba155c5f227,0.825177,0,1613,1,0.06,3.1,4,5,4,0,0.0,104
27585,content_d831e97bab46,0.788270,0,1360,0,0.00,6.1,15,21,13,0,0.0,104
9192,content_59c260251c82,0.786491,0,8795,2,0.02,34.9,5,6,5,0,0.0,20
22928,content_929aa622b6a0,0.781423,0,11301,3,0.03,2.4,7,7,7,0,0.0,104
18895,content_b50209ec3c8f,0.781290,0,3273,2,0.06,25.8,4,6,3,0,0.0,13



TOP FALSE NEGATIVES


,content_id,model_probability,target_refresh,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,pageviews_90d,users_90d,engaged_sessions_90d,engagement_rate,days_since_last_update
28644,content_7f6b72253c41,0.499492,1,37,0,0.00,2.8,2,1,2,0,0.0,20
5514,content_679828cc0bfe,0.499217,1,27,0,0.00,12.1,1,1,1,0,0.0,20
14440,content_86003ed8aa34,0.499031,1,341,4,1.17,13.3,5,6,5,1,20.0,20
24600,content_bfec24fd6a60,0.498703,1,233,3,1.29,4.6,4,4,4,0,0.0,20
18678,content_d5a9fc0e9ea6,0.498233,1,35,0,0.00,2.7,1,1,1,0,0.0,20


### Error interpretation

The false positives are items that the model ranked highly for refresh but that did not meet the proxy target. These represent potential review work that would not have been selected by the proxy definition.

The false negatives are items that met the proxy target but were not ranked highly enough by the model. These represent potentially missed opportunities under the defined proxy.

These errors show that the model is not a perfect classifier. Precision@50 measures the quality of the highest-priority recommendations, so the model should be used to support human review rather than automatically determine which pages must be refreshed.

The error examples are based on the proxy target and therefore do not establish whether the recommended editorial action would produce a future SEO improvement.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.